# atari_50m_endpoint

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from analysis.curves import episode_average, return_curve, subsample
from analysis.stats import mean_ci, min_max_normalize
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Each setting's component per agent, and each figure set's agents in drawing
# order with their colors. Endpoint comes last so it is drawn on top. Agents
# with no stored runs yet are left out of every plot.
COMPONENTS = {
    "medium": {
        "Large": "ddqn_atari",
        "Small": "ddqn_medium_atari",
        "Small 10-step": "ddqn_medium_nstep_atari",
        "Unanchored": "unanchored_atari",
        "Reservoir": "reservoir_atari",
        "Endpoint": "endpoint_atari",
    },
    "small": {
        "Large": "ddqn_atari",
        "Small": "ddqn_small_atari",
        "Small 10-step": "ddqn_small_nstep_atari",
        "Unanchored": "unanchored_small_atari",
        "Reservoir": "reservoir_small_atari",
        "Endpoint": "endpoint_small_atari",
    },
}
FIGURES = {
    "paper": {
        "Large": "black",
        "Small": "tab:blue",
        "Small 10-step": "tab:purple",
        "Reservoir": "#125A56",
        "Endpoint": "tab:red",
    },
    "poster": {
        "Large": "black",
        "Small": "tab:blue",
        "Unanchored": "#a90064",
        "Endpoint": "#009f16",
    },
}
# Legend and bar order.
ORDER = ["Endpoint", "Large", "Small", "Small 10-step", "Unanchored", "Reservoir"]
# Memory relative to the 1M large-recency buffer.
TITLES = {"medium": "Atari 10x Smaller", "small": "Atari 50x Smaller"}

component = EXPERIMENT.component("ddqn_atari")
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP

In [ ]:
# Load each component's stored runs per game once. Compute its mean return per
# game with a 95% bootstrap CI, and each run's episode-average return for the
# bar plots.
names = {name for agents in COMPONENTS.values() for name in agents.values()}
curves = {}
episode_scores = {}
for name in names:
    for game in GAMES:
        runs = load_results(EXPERIMENT, name, where={"ENV_HYPERS.GAME": game})
        if runs is None:
            continue
        curve = return_curve(runs.reward, runs.done)
        timesteps, returns = subsample(curve)
        curves[name, game] = (timesteps * FRAMESKIP, mean_ci(returns), len(returns))
        episode_scores[name, game] = episode_average(runs.reward, runs.done)


def present(figure, setting):
    agents = COMPONENTS[setting]
    return [
        agent
        for agent in FIGURES[figure]
        if any((agents[agent], game) in curves for game in GAMES)
    ]

In [ ]:
# Style for the all-games grids, laid out as in atari_50m_deterministic.
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP
GRID_STYLE = {
    "figure.figsize": (16, 9.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (3, 4)
PANELS = {  # (column, row) of each game's panel
    "battle_zone": (0, 0),
    "double_dunk": (0, 1),
    "name_this_game": (0, 2),
    "phoenix": (1, 0),
    "qbert": (1, 1),
    "pong": (1, 2),
    "breakout": (2, 0),
    "seaquest": (2, 1),
    "centipede": (2, 2),
    "ms_pacman": (3, 0),
    "beam_rider": (3, 1),
    "space_invaders": (3, 2),
}
PANEL_ASPECT = 3 / 4
LINE_WIDTH = 1.75
BAND_ALPHA = 0.2
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.06, 1, 1), "w_pad": 0.2, "h_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.055, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {
    "loc": "upper center",
    "bbox_to_anchor": (0.5, 0.035),
    "handlelength": 1,
    "handletextpad": 0.4,
    "columnspacing": 1.2,
}
GRID_LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 12}


def title(game):
    return game.replace("_", " ").title()

In [ ]:
# Plot every game in one grid per figure set and setting, with the legend on a
# single row.
grids = {}
for figure, colors in FIGURES.items():
    for setting, agents in COMPONENTS.items():
        shown = present(figure, setting)
        if not shown:
            continue
        with plt.rc_context(GRID_STYLE):
            grid, axes = plt.subplots(*GRID_SHAPE)
            for game, (column, row) in PANELS.items():
                ax = axes[row, column]
                ax.set_box_aspect(PANEL_ASPECT)
                drawn = [agent for agent in shown if (agents[agent], game) in curves]
                if not drawn:
                    ax.axis("off")
                    continue
                for agent in drawn:
                    frames, (mean, low, high), _ = curves[agents[agent], game]
                    color = colors[agent]
                    ax.fill_between(frames, low, high, color=color, alpha=BAND_ALPHA)
                    ax.plot(frames, mean, color=color, linewidth=LINE_WIDTH)
                ax.set_title(title(game))
                ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
            grid.tight_layout(**GRID_LAYOUT)
            grid.supxlabel(**GRID_XLABEL)
            grid.supylabel(**GRID_YLABEL)
            labels = [agent for agent in ORDER if agent in shown]
            handles = [
                Line2D([], [], color=colors[agent], **GRID_LEGEND_MARKER)
                for agent in labels
            ]
            grid.legend(handles, labels, ncol=len(labels), **GRID_LEGEND)
        grids[f"{figure}_grid_{setting}"] = grid
plt.show()

In [ ]:
# Style for the normalized-score bar plots.
BAR_STYLE = {
    "figure.figsize": (14, 7),
    "font.family": "Times New Roman",
    "font.size": 32,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 32,
    "axes.labelsize": 32,
    "ytick.labelsize": 32,
}
BAR_WIDTH = 0.6  # a quarter thinner than the default 0.8
BAR_YLABEL = "Normalized Average Return"
BAR_YLABEL_PAD = 0
BAR_ASPECT = 1
# Each panel's y-axis top, which is also its upper tick.
BAR_TOPS = {"medium": 0.7, "small": 0.8}
# Bars are named inside, reading up from their base; a bar too short for that
# carries its name on top instead, one word per line.
BAR_INSIDE_LABEL = {"rotation": 90, "ha": "center", "va": "bottom", "color": "white"}
BAR_INSIDE_MIN = 0.3
# Per-figure overrides: bar order, and figures whose names always sit inside.
BAR_ORDERS = {"poster": ["Large", "Small", "Unanchored", "Endpoint"]}
# Poster names all sit inside; a smaller font fits "Unanchored" in its short bar.
BAR_ALWAYS_INSIDE = {"poster"}
BAR_LABEL_SIZES = {"Unanchored": 24}
BAR_TOP_LABEL = {"ha": "center", "va": "bottom"}
BAR_LABEL_GAP = 0.01

In [ ]:
# Normalize each game's runs onto [0, 1] with bounds shared by a panel's agents,
# then average over games and seeds.
def normalized_means(figure, setting):
    agents = COMPONENTS[setting]
    order = BAR_ORDERS.get(figure, ORDER)
    shown = [agent for agent in order if agent in present(figure, setting)]
    complete = [
        game
        for game in GAMES
        if all((agents[agent], game) in episode_scores for agent in shown)
    ]
    if not complete:
        return {}
    totals = dict.fromkeys(shown, 0.0)
    for game in complete:
        stacks = [episode_scores[agents[agent], game] for agent in shown]
        for agent, stack in zip(shown, min_max_normalize(stacks), strict=True):
            totals[agent] += float(np.mean(stack))
    return {agent: total / len(complete) for agent, total in totals.items()}


bar_figures = {}
for figure, colors in FIGURES.items():
    panels = {setting: normalized_means(figure, setting) for setting in COMPONENTS}
    if not any(panels.values()):
        continue
    with plt.rc_context(BAR_STYLE):
        fig, axes = plt.subplots(1, len(panels))
        for column, (ax, (setting, means)) in enumerate(
            zip(axes, panels.items(), strict=True)
        ):
            ax.set_box_aspect(BAR_ASPECT)
            for position, (agent, mean) in enumerate(means.items()):
                ax.bar(position, mean, width=BAR_WIDTH, color=colors[agent])
                if figure in BAR_ALWAYS_INSIDE or mean >= BAR_INSIDE_MIN:
                    size = BAR_LABEL_SIZES.get(agent)
                    inside = (position, BAR_LABEL_GAP, agent)
                    ax.text(*inside, fontsize=size, **BAR_INSIDE_LABEL)
                else:
                    label = agent.replace(" ", "\n")
                    top_label = (position, mean + BAR_LABEL_GAP, label)
                    ax.text(*top_label, **BAR_TOP_LABEL)
            ax.set_title(TITLES[setting])
            ax.set_xticks([])
            top = BAR_TOPS[setting]
            ax.set_yticks([0, top])
            ax.set_ylim(0, top)
            if column == 0:
                ax.set_ylabel(BAR_YLABEL, labelpad=BAR_YLABEL_PAD)
        fig.tight_layout()
    bar_figures[f"{figure}_bars"] = fig
plt.show()

In [ ]:
# Save every figure as a PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in {**grids, **bar_figures}.items():
    for suffix in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{suffix}", bbox_inches="tight")
print(f"saved {len(grids) + len(bar_figures)} figure(s) to {PLOTS_DIR}")